# Train and Evaluate All 14 Classifiers

Trains and evaluates all 14 classifiers (8 base classifiers + 6 ensemble classifiers) on the modeling dataset, using the hyperparameter grids, preprocessing rules, 5-fold stratified CV, and train/test split.

In [1]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, BaggingClassifier, ExtraTreesClassifier,
                               AdaBoostClassifier, StackingClassifier)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
HORIZONS = [1, 2, 3, 4, 5]

pd.set_option('display.max_columns', None)

## 1. Load modeling dataset + feature list

In [2]:
df = pd.read_csv('modeling_dataset.csv')
COMORBID_FEATURE_COLS = [c for c in df.columns if c not in (
    ['person_id', 'sex', 'age', 'postcode', 'rurality', 'ses_irsd_decile', 'ses_missing',
     'incident_cvd', 'years_followup', 'split',
     'baseline_n_diagnoses', 'baseline_n_procedures', 'baseline_n_claims',
     'baseline_n_episodes', 'baseline_history_days']
    + [f'label_{h}y' for h in HORIZONS] + [f'eligible_{h}y' for h in HORIZONS]
)]
IS_GEO = 'rurality' in df.columns

UTILISATION_COLS = ['baseline_n_diagnoses', 'baseline_n_procedures', 'baseline_n_claims',
                     'baseline_n_episodes', 'baseline_history_days']
if IS_GEO:
    NUMERIC_COLS = ['age'] + UTILISATION_COLS + ['ses_irsd_decile', 'ses_missing']
    CATEGORICAL_COLS = ['sex', 'rurality']
else:
    NUMERIC_COLS = ['age'] + UTILISATION_COLS
    CATEGORICAL_COLS = ['sex']

FEATURE_COLS = CATEGORICAL_COLS + NUMERIC_COLS + COMORBID_FEATURE_COLS
print(f'{len(FEATURE_COLS)} features ({len(COMORBID_FEATURE_COLS)} comorbidity-derived): {FEATURE_COLS}')

def make_preprocessor(scaled):
    num_step = StandardScaler() if scaled else 'passthrough'
    return ColumnTransformer([
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), CATEGORICAL_COLS),
        ('num', num_step, NUMERIC_COLS),
        ('bin', 'passthrough', COMORBID_FEATURE_COLS),
    ])

19 features (12 comorbidity-derived): ['sex', 'age', 'baseline_n_diagnoses', 'baseline_n_procedures', 'baseline_n_claims', 'baseline_n_episodes', 'baseline_history_days', 'cat_Renal', 'cat_FluidElectrolyte', 'cat_Metabolic', 'cat_Respiratory', 'cat_MentalHealthSubstance', 'cat_Musculoskeletal', 'cat_Hematologic', 'cat_Hepatic', 'cat_Neurologic', 'cat_Oncologic', 'cat_EndocrineOther', 'cat_ConstitutionalFrailty']


## 2. Model configs — identical grids to the baseline variant

In [3]:
MODEL_CONFIGS = {
    'logreg': dict(name='Logistic Regression', cls=LogisticRegression, scaled=True,
                   fixed_kwargs=dict(solver='saga', max_iter=5000, random_state=RANDOM_STATE),
                   param_grid={'C': [0.01, 0.1, 1, 10], 'penalty': ['l1', 'l2'], 'class_weight': [None, 'balanced']}),
    'rf': dict(name='Random Forest', cls=RandomForestClassifier, scaled=False,
               fixed_kwargs=dict(random_state=RANDOM_STATE, n_jobs=-1),
               param_grid={'n_estimators': [100, 300, 500], 'max_depth': [None, 5, 10],
                           'min_samples_leaf': [1, 5, 10], 'class_weight': [None, 'balanced']}),
    'xgboost': dict(name='XGBoost', cls=XGBClassifier, scaled=False,
                     fixed_kwargs=dict(random_state=RANDOM_STATE, eval_metric='logloss'),
                     param_grid={'n_estimators': [100, 300], 'max_depth': [3, 5, 7],
                                 'learning_rate': [0.01, 0.1], 'subsample': [0.8, 1.0]},
                     needs_scale_pos_weight=True),
    'lightgbm': dict(name='LightGBM', cls=LGBMClassifier, scaled=False,
                      fixed_kwargs=dict(random_state=RANDOM_STATE, class_weight='balanced', verbose=-1),
                      param_grid={'n_estimators': [100, 300], 'max_depth': [3, 5, -1],
                                  'learning_rate': [0.01, 0.1], 'num_leaves': [15, 31]}),
    'svm': dict(name='SVM', cls=SVC, scaled=True,
                fixed_kwargs=dict(probability=True, random_state=RANDOM_STATE),
                param_grid={'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf'], 'class_weight': [None, 'balanced']}),
    'dtree': dict(name='Decision Tree', cls=DecisionTreeClassifier, scaled=False,
                   fixed_kwargs=dict(random_state=RANDOM_STATE),
                   param_grid={'max_depth': [None, 5, 10], 'min_samples_leaf': [5, 10, 20],
                               'criterion': ['gini', 'entropy'], 'class_weight': [None, 'balanced']}),
    'nb': dict(name='Naive Bayes', cls=GaussianNB, scaled=True, fixed_kwargs=dict(),
               param_grid={'var_smoothing': [1e-9, 1e-8, 1e-7], 'priors': [None, [0.5, 0.5]]}),
    'knn': dict(name='KNN', cls=KNeighborsClassifier, scaled=True, fixed_kwargs=dict(),
                param_grid={'n_neighbors': [5, 11, 21, 31], 'weights': ['uniform', 'distance'],
                            'metric': ['euclidean', 'manhattan']}),
    'bag_dt': dict(name='Bagging (DT base)', cls=BaggingClassifier, scaled=False,
                    fixed_kwargs=dict(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
                                       random_state=RANDOM_STATE, n_jobs=-1),
                    param_grid={'n_estimators': [50, 100, 200], 'max_samples': [0.5, 1.0],
                                'estimator__class_weight': [None, 'balanced']}),
    'extratrees': dict(name='Extra Trees', cls=ExtraTreesClassifier, scaled=False,
                        fixed_kwargs=dict(random_state=RANDOM_STATE, n_jobs=-1),
                        param_grid={'n_estimators': [100, 300, 500], 'max_depth': [None, 5, 10],
                                    'min_samples_leaf': [1, 5, 10], 'class_weight': [None, 'balanced']}),
    'bag_knn': dict(name='Bagging (KNN base)', cls=BaggingClassifier, scaled=True,
                     fixed_kwargs=dict(estimator=KNeighborsClassifier(), random_state=RANDOM_STATE, n_jobs=-1),
                     param_grid={'n_estimators': [50, 100, 200], 'max_samples': [0.5, 1.0],
                                 'estimator__n_neighbors': [5, 11, 21]}),
    'adaboost': dict(name='AdaBoost', cls=AdaBoostClassifier, scaled=False,
                      fixed_kwargs=dict(random_state=RANDOM_STATE),
                      param_grid={'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 1.0]}),
    'stack_diverse': dict(name='Stacking (diverse)', cls=StackingClassifier, scaled=True,
                           fixed_kwargs=dict(
                               estimators=[
                                   ('lr', LogisticRegression(solver='saga', max_iter=5000, class_weight='balanced', random_state=RANDOM_STATE)),
                                   ('rf', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
                                   ('svm', SVC(probability=True, class_weight='balanced', random_state=RANDOM_STATE)),
                               ],
                               final_estimator=LogisticRegression(max_iter=5000, random_state=RANDOM_STATE), cv=5, n_jobs=-1),
                           param_grid={'final_estimator__C': [0.1, 1, 10], 'passthrough': [True, False]}),
    'stack_boosting': dict(name='Stacking (boosting)', cls=StackingClassifier, scaled=True,
                            fixed_kwargs=dict(
                                estimators=[
                                    ('xgb', XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss')),
                                    ('lgbm', LGBMClassifier(class_weight='balanced', random_state=RANDOM_STATE, verbose=-1)),
                                    ('ada', AdaBoostClassifier(random_state=RANDOM_STATE)),
                                ],
                                final_estimator=LogisticRegression(max_iter=5000, random_state=RANDOM_STATE), cv=5, n_jobs=-1),
                            param_grid={'final_estimator__C': [0.1, 1, 10], 'passthrough': [True, False]}),
}

## 3. Train + tune + evaluate every model x horizon

In [4]:
all_results = {}

for prefix, cfg in MODEL_CONFIGS.items():
    preprocessor = make_preprocessor(cfg['scaled'])
    results = []
    for h in HORIZONS:
        elig_col, label_col = f'eligible_{h}y', f'label_{h}y'
        sub = df[df[elig_col]].copy()
        train, test = sub[sub['split'] == 'train'], sub[sub['split'] == 'test']
        X_train, y_train = train[FEATURE_COLS], train[label_col].astype(int)
        X_test, y_test = test[FEATURE_COLS], test[label_col].astype(int)

        init_kwargs = dict(cfg['fixed_kwargs'])
        if cfg.get('needs_scale_pos_weight'):
            init_kwargs['scale_pos_weight'] = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

        pipe = Pipeline([('preprocess', preprocessor), ('clf', cfg['cls'](**init_kwargs))])
        param_grid = {f'clf__{k}': v for k, v in cfg['param_grid'].items()}
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        grid = GridSearchCV(pipe, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, refit=True)
        grid.fit(X_train, y_train)
        best = grid.best_estimator_

        test_proba = best.predict_proba(X_test)[:, 1]
        pred = (test_proba >= 0.5).astype(int)

        res = dict(horizon=f'{h}y', n_train=len(X_train), n_test=len(X_test),
                   train_event_rate=y_train.mean(), test_event_rate=y_test.mean(),
                   best_params=grid.best_params_, cv_auc_mean=grid.best_score_,
                   test_auc=roc_auc_score(y_test, test_proba),
                   test_accuracy=accuracy_score(y_test, pred),
                   test_precision=precision_score(y_test, pred, zero_division=0),
                   test_recall=recall_score(y_test, pred, zero_division=0),
                   test_f1=f1_score(y_test, pred, zero_division=0))
        results.append(res)
        joblib.dump(best, f'{prefix}_{h}y_model.joblib')
        print(f"{cfg['name']:22s} {h}y: AUC={res['test_auc']:.3f}  acc={res['test_accuracy']:.3f}  f1={res['test_f1']:.3f}")

    results_df = pd.DataFrame(results)
    results_df.to_csv(f'{prefix}_results.csv', index=False)
    all_results[prefix] = results_df

print('\nAll 14 classifiers trained.')

Logistic Regression    1y: AUC=0.695  acc=0.666  f1=0.229


Logistic Regression    2y: AUC=0.706  acc=0.661  f1=0.339


Logistic Regression    3y: AUC=0.689  acc=0.636  f1=0.411


Logistic Regression    4y: AUC=0.649  acc=0.601  f1=0.430


Logistic Regression    5y: AUC=0.643  acc=0.630  f1=0.560


...

Stacking (boosting)    2y: AUC=0.688  acc=0.848  f1=0.000


Stacking (boosting)    3y: AUC=0.680  acc=0.793  f1=0.096


Stacking (boosting)    4y: AUC=0.655  acc=0.720  f1=0.355


Stacking (boosting)    5y: AUC=0.653  acc=0.620  f1=0.537



All 14 classifiers trained.


## 4. Summary: best AUC per model, all horizons

In [5]:
summary_rows = []
for prefix, cfg in MODEL_CONFIGS.items():
    rdf = all_results[prefix]
    for _, row in rdf.iterrows():
        summary_rows.append(dict(model=cfg['name'], horizon=row['horizon'], test_auc=row['test_auc']))
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('classifier_summary_auc.csv', index=False)
summary_df.pivot(index='model', columns='horizon', values='test_auc')[['1y', '2y', '5y']].round(3)

horizon,1y,2y,5y
model,,,
AdaBoost,0.697,0.689,0.640
Bagging (DT base),0.671,0.679,0.605
Bagging (KNN base),0.645,0.672,0.583
Decision Tree,0.643,0.618,0.580
Extra Trees,0.694,0.691,0.627
KNN,0.681,0.682,0.582
LightGBM,0.667,0.675,0.640
Logistic Regression,0.695,0.706,0.643
Naive Bayes,0.673,0.683,0.660
